# DDP Phase 2 — DermaMNIST: PRIME ensemble (GPU notebook)

**Prior-corrected, Resolution-matched, heterogeneous EfficientNet ensemble with TTA.**

Runs the same pipeline as `run/train_v2.py` + `run/ensemble_v2.py` on a Colab/Kaggle GPU.
Set `RUNTIME -> GPU` first. Results (probabilities, checkpoints, JSON, figures) are written to
`WORK_DIR` (Google Drive if mounted) so the run can be resumed.

Members trained by default (edit `MEMBERS` below):
1. `b0_96`   EfficientNet-B0, 64-px data upsampled to 96 px (reproduces the July baseline recipe)
2. `v2b0_96` EfficientNetV2-B0, 96 px
3. `b0_128n` EfficientNet-B0 on the **native 128-px MedMNIST+ release** (dermamnist_128.npz)
4. `v2b1_128n` EfficientNetV2-B1 on native 128 px

Members 3–4 use higher-resolution *data*, so they must be reported against the MedMNIST+ 128-px
column, not the 64-px column. Members 1–2 are the like-for-like comparison with the July 80.0%.

In [ ]:
import os, json, time, hashlib, urllib.request
import numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report
print(tf.__version__, tf.config.list_physical_devices('GPU'))

# ---- where to keep everything (resume-safe) ----
try:
    from google.colab import drive; drive.mount('/content/drive')
    WORK_DIR = '/content/drive/MyDrive/DDP_Phase2'
except Exception:
    WORK_DIR = './DDP_Phase2'
os.makedirs(WORK_DIR, exist_ok=True); print('WORK_DIR =', WORK_DIR)

ZENODO = 'https://zenodo.org/records/10519652/files/{}?download=1'   # MedMNIST+ record
MD5 = {'dermamnist_64.npz': 'b70a2f5635c6199aeaa28c31d7202e1f', 'dermamnist_128.npz': '2defd784463fa5243564e855ed717de1'}
def fetch(name):
    path = os.path.join(WORK_DIR, name)
    if not os.path.exists(path):
        print('downloading', name); urllib.request.urlretrieve(ZENODO.format(name), path)
    h = hashlib.md5(open(path, 'rb').read()).hexdigest()
    assert h == MD5[name], f'MD5 mismatch for {name}: {h}'
    return path
DATA = {64: fetch('dermamnist_64.npz')}
try: DATA[128] = fetch('dermamnist_128.npz')
except Exception as e: print('128-px data unavailable:', e)

In [ ]:
NUM_CLASSES = 7
CLASS_NAMES = ['Actinic Ker.', 'Basal Cell Ca.', 'Benign Ker.', 'Dermatofibroma', 'Melanoma', 'Nevi', 'Vascular']
BACKBONES = {'B0': keras.applications.EfficientNetB0, 'B1': keras.applications.EfficientNetB1,
             'B2': keras.applications.EfficientNetB2, 'B3': keras.applications.EfficientNetB3,
             'V2B0': keras.applications.EfficientNetV2B0, 'V2B1': keras.applications.EfficientNetV2B1,
             'V2B2': keras.applications.EfficientNetV2B2, 'V2S': keras.applications.EfficientNetV2S}

# name: (backbone, native data resolution, model input size, batch, unfreeze_top, epochs_head, epochs_ft)
MEMBERS = {
    'b0_96':     ('B0',   64,  96,  32, 60, 15, 30),
    'v2b0_96':   ('V2B0', 64,  96,  32, 60, 15, 30),
    'b0_128n':   ('B0',   128, 128, 32, 80, 15, 30),
    'v2b1_128n': ('V2B1', 128, 128, 32, 80, 15, 30),
}
LABEL_SMOOTH, DROPOUT, LR_HEAD, LR_FT, LR_FT_MIN, PAT_HEAD, PAT_FT, TTA_N, SEED = 0.05, 0.40, 1e-3, 1e-4, 1e-6, 5, 8, 8, 42

class SmoothedCE(keras.losses.Loss):      # label-smoothed cross-entropy (gamma=0 case of the July focal loss)
    def __init__(self, label_smoothing=LABEL_SMOOTH, name='smoothed_ce', **kw):
        super().__init__(name=name, **kw); self.label_smoothing = label_smoothing
    def call(self, y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_sm = tf.one_hot(y_true, NUM_CLASSES) * (1 - self.label_smoothing) + self.label_smoothing / NUM_CLASSES
        return -tf.reduce_sum(y_sm * tf.math.log(tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)), -1)
    def get_config(self): c = super().get_config(); c.update(label_smoothing=self.label_smoothing); return c

augment = keras.Sequential([layers.RandomFlip('horizontal_and_vertical'), layers.RandomRotation(0.25),
                            layers.RandomZoom(0.20, 0.20), layers.RandomTranslation(0.10, 0.10),
                            layers.RandomContrast(0.20), layers.RandomBrightness(0.20)])
tta_aug = keras.Sequential([layers.RandomFlip('horizontal_and_vertical'), layers.RandomRotation(0.10),
                            layers.RandomZoom(0.05), layers.RandomContrast(0.10)])

def make_ds(x, y, img, batch, training):
    ds = tf.data.Dataset.from_tensor_slices((x.astype(np.float32), y.flatten().astype(np.int32)))
    if training: ds = ds.shuffle(2048, seed=SEED, reshuffle_each_iteration=True)
    def prep(im, lb):
        im = tf.image.resize(im, [img, img])
        if training: im = augment(im, training=True)
        return tf.clip_by_value(im, 0., 255.), lb
    return ds.map(prep, num_parallel_calls=tf.data.AUTOTUNE).batch(batch).prefetch(tf.data.AUTOTUNE)

def build(backbone_name, img):
    bb = BACKBONES[backbone_name](include_top=False, weights='imagenet', input_shape=(img, img, 3)); bb.trainable = False
    inp = layers.Input((img, img, 3)); x = bb(inp, training=False)
    x = layers.Add()([layers.GlobalAveragePooling2D()(x), layers.GlobalMaxPooling2D()(x)])   # dual pooling head
    x = layers.BatchNormalization()(x)
    for units, dr in ((512, DROPOUT), (128, DROPOUT / 2)):
        x = layers.Dense(units, use_bias=False)(x); x = layers.BatchNormalization()(x)
        x = layers.Activation('swish')(x); x = layers.Dropout(dr)(x)
    return keras.Model(inp, layers.Dense(NUM_CLASSES, activation='softmax')(x)), bb

def predict_tta(model, x, img, n=TTA_N):
    xr = tf.image.resize(tf.cast(x, tf.float32), [img, img]).numpy()
    std = model.predict(xr, batch_size=128, verbose=0); tta = std.copy()
    for _ in range(n):
        tta += model.predict(np.clip(tta_aug(tf.constant(xr), training=True).numpy(), 0, 255), batch_size=128, verbose=0)
    return std, tta / (n + 1)

In [ ]:
def train_member(name):
    bbname, res, img, batch, unfreeze, ep_head, ep_ft = MEMBERS[name]
    out = os.path.join(WORK_DIR, name); os.makedirs(out, exist_ok=True)
    if os.path.exists(os.path.join(out, 'results.json')):
        print('skip', name, '(done)'); return json.load(open(os.path.join(out, 'results.json')))
    if res not in DATA: print('skip', name, '(no data at', res, 'px)'); return None
    d = np.load(DATA[res]); tf.random.set_seed(SEED); np.random.seed(SEED)
    tr = make_ds(d['train_images'], d['train_labels'], img, batch, True)
    va = make_ds(d['val_images'], d['val_labels'], img, batch, False)
    y_val, y_te = d['val_labels'].flatten(), d['test_labels'].flatten()
    ckpt = os.path.join(out, 'best_model.keras'); metrics = [keras.metrics.SparseCategoricalAccuracy(name='accuracy')]
    model, bb = build(bbname, img)
    # phase 1: head
    model.compile(keras.optimizers.Adam(LR_HEAD), SmoothedCE(), metrics)
    model.fit(tr, validation_data=va, epochs=ep_head, verbose=2, callbacks=[
        keras.callbacks.EarlyStopping('val_accuracy', patience=PAT_HEAD, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau('val_loss', factor=0.5, patience=2, min_lr=1e-6)])
    # phase 2: fine-tune top layers, BN frozen, cosine AdamW
    bb.trainable = True; cut = len(bb.layers) - unfreeze
    for i, l in enumerate(bb.layers): l.trainable = i >= cut and not isinstance(l, layers.BatchNormalization)
    sched = keras.optimizers.schedules.CosineDecay(LR_FT, ep_ft * len(tr), alpha=LR_FT_MIN / LR_FT)
    model.compile(keras.optimizers.AdamW(sched, weight_decay=1e-4), SmoothedCE(), metrics)
    hist = model.fit(tr, validation_data=va, epochs=ep_ft, verbose=2, callbacks=[
        keras.callbacks.EarlyStopping('val_accuracy', patience=PAT_FT, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint(ckpt, monitor='val_accuracy', save_best_only=True)])
    model.save(ckpt)
    # probabilities for val + test (TTA)
    vs, vt = predict_tta(model, d['val_images'], img); ts, tt = predict_tta(model, d['test_images'], img)
    for arr, fn in ((vs, 'val_std_probs'), (vt, 'val_tta_probs'), (ts, 'test_std_probs'), (tt, 'test_tta_probs'),
                    (y_val, 'y_val'), (y_te, 'y_test')): np.save(os.path.join(out, fn + '.npy'), arr)
    res_ = dict(member=name, backbone=bbname, data_res=res, img=img,
                val_tta_top1=accuracy_score(y_val, vt.argmax(1)), std_top1=accuracy_score(y_te, ts.argmax(1)),
                tta_top1=accuracy_score(y_te, tt.argmax(1)), tta_balanced=balanced_accuracy_score(y_te, tt.argmax(1)),
                history={k: [float(v) for v in vals] for k, vals in hist.history.items()})
    json.dump(res_, open(os.path.join(out, 'results.json'), 'w'), indent=2)
    print(name, {k: round(v, 4) for k, v in res_.items() if isinstance(v, float)})
    print(classification_report(y_te, tt.argmax(1), target_names=CLASS_NAMES, digits=4))
    return res_

results = {n: train_member(n) for n in MEMBERS}

In [ ]:
# ---- Ensemble (validation-tuned weights) + prior correction (logit adjustment) ----
import itertools
from sklearn.metrics import roc_auc_score, f1_score
done = [n for n in MEMBERS if os.path.exists(os.path.join(WORK_DIR, n, 'test_tta_probs.npy'))]
V = [np.load(os.path.join(WORK_DIR, n, 'val_tta_probs.npy')) for n in done]
T = [np.load(os.path.join(WORK_DIR, n, 'test_tta_probs.npy')) for n in done]
y_val = np.load(os.path.join(WORK_DIR, done[0], 'y_val.npy')); y_te = np.load(os.path.join(WORK_DIR, done[0], 'y_test.npy'))
prior = np.bincount(np.load(DATA[64])['train_labels'].flatten(), minlength=7) / 7007
grid = np.arange(0, 1.0001, 0.1); best = (-1, None)
for w in itertools.product(grid, repeat=len(done)):
    if abs(sum(w) - 1) > 1e-6: continue
    acc = accuracy_score(y_val, sum(wi * v for wi, v in zip(w, V)).argmax(1))
    if acc > best[0] + 1e-9: best = (acc, w)
w = best[1]; pv = sum(wi * v for wi, v in zip(w, V)); pt = sum(wi * t for wi, t in zip(w, T))
print('members', done, 'weights', w, 'val top-1 %.4f' % best[0])
adj = lambda p, tau: (np.log(np.clip(p, 1e-9, 1)) - tau * np.log(prior)).argmax(1)
taus = np.round(np.arange(0, 1.51, 0.05), 2)
vt1 = np.array([accuracy_score(y_val, adj(pv, t)) for t in taus]); vb = np.array([balanced_accuracy_score(y_val, adj(pv, t)) for t in taus])
tau_B = float(taus[np.where(vt1 >= vt1.max() - 0.005)[0].max()]); tau_C = float(taus[vb.argmax()])
summary = {}
for name, tau in (('A_accuracy', 0.0), ('B_screening', tau_B), ('C_balanced', tau_C)):
    pred = adj(pt, tau)
    summary[name] = dict(tau=tau, top1=accuracy_score(y_te, pred), balanced=balanced_accuracy_score(y_te, pred),
                         macro_f1=f1_score(y_te, pred, average='macro'),
                         auc=roc_auc_score(y_te, pt / pt.sum(1, keepdims=True), multi_class='ovr'))
    print(name, {k: round(v, 4) for k, v in summary[name].items()})
json.dump(dict(members=done, weights=list(w), tau_B=tau_B, tau_C=tau_C, points=summary,
               member_results={n: {k: v for k, v in (results[n] or {}).items() if k != 'history'} for n in done}),
          open(os.path.join(WORK_DIR, 'ensemble_results.json'), 'w'), indent=2)
print('saved', os.path.join(WORK_DIR, 'ensemble_results.json'))